In [28]:
import numpy as np
import pandas as pd
from datetime import date, timedelta
from sqlalchemy import create_engine
from pandas.tseries.offsets import BDay

engine = create_engine("sqlite:///c:\\ruby\\port_lite\\db\\development.sqlite3")
conlite = engine.connect()

data_path = "../data/"
csv_path = "\\Users\\User\\iCloudDrive\\"
box_path = "\\Users\\User\\Dropbox\\"
one_path = "\\Users\\User\\OneDrive\\Documents\\Data\\"
osd_path = "\\Users\\User\\OneDrive\\Documents\\obsidian-git-sync\\Data\\"

today = date.today()
today

datetime.date(2023, 2, 7)

### Yesterday = Last business day

In [29]:
# convert the timedelta object to a BusinessDay object
num_business_days = BDay(1)
yesterday = today - num_business_days
yesterday = yesterday.date()
print(f'today: {today}')
print(f'yesterday: {yesterday}')

today: 2023-02-07
yesterday: 2023-02-06


In [30]:
cols = 'fm_date to_date fm_price to_price qty max_price min_price percent status'.split()

format_dict = {
    'fm_price':'{:.2f}','to_price':'{:.2f}','diff':'{:.2f}',
    'max_price':'{:.2f}','min_price':'{:.2f}',
    'volume':'{:,.2f}','beta':'{:,.2f}',
    'pct':'{:,.2f}%','percent':'{:,.2f}%',   
    'fm_date':'{:%Y-%m-%d}','to_date':'{:%Y-%m-%d}',
    'created_at':'{:%Y-%m-%d}','updated_at':'{:%Y-%m-%d}',
    
    'qty':'{:,}','available_qty':'{:,}',
    'cost':'{:.2f}','buy_target':'{:.2f}','sell_target':'{:.2f}',
}

### Begin of Tables in the process

In [31]:
sql = """
SELECT * 
FROM sales 
ORDER BY id DESC LIMIT 1
"""
tmp = pd.read_sql(sql, conlite)
tmp["fm_date"] = pd.to_datetime(tmp["fm_date"])
tmp["to_date"] = pd.to_datetime(tmp["to_date"])
tmp["created_at"] = pd.to_datetime(tmp["created_at"])
tmp["updated_at"] = pd.to_datetime(tmp["updated_at"])
tmp.style.format(format_dict)

,id,name,fm_date,to_date,days,fm_price,to_price,diff,pct,ttl_spread,avg_spread,max_price,min_price,qty,created_at,updated_at,latest_date_id
0,1216121,PTT,2023-02-06,2023-02-06,0,33.00,32.75,-0.25,-0.76%,-1,0,34.00,32.50,"34,781,194",2023-02-07,2023-02-07,1


In [32]:
names = tmp["name"]
type(names)

pandas.core.series.Series

In [33]:
name = names.to_string(index=False)
type(name)

str

In [34]:
sql = """
SELECT * 
FROM stocks
WHERE name = '%s'
"""
sql = sql % name
print(sql)
tmp = pd.read_sql(sql, conlite)
tmp.style.format(format_dict)


SELECT * 
FROM stocks
WHERE name = 'PTT'



,id,name,max_price,min_price,status,buy_target,sell_target,volume,beta,cost,qty,buy_spread,sell_spread,available_qty,bl,sh,reason,market
0,9777,PTT,0.00,0.00,O,32.00,0.00,0.00,0.00,0.00,"3,600",-19,7,0,0,0,XXX,SET50


### End of Tables in the process

In [35]:
sql = """
SELECT DISTINCT a.name
FROM sales a 
WHERE to_date = '%s'
ORDER BY a.name
"""
sql = sql % yesterday
print(sql)
tmp = pd.read_sql(sql, conlite)
tmp


SELECT DISTINCT a.name
FROM sales a 
WHERE to_date = '2023-02-06'
ORDER BY a.name



,name
0,BANPU
1,CK
2,CKP
3,COM7
4,CPF
5,DIF
6,IVL
7,JASIF
8,JMART
9,JMT


In [36]:
sql = """
SELECT a.name,fm_date,to_date,fm_price,to_price,
a.qty,a.max_price,a.min_price,t.status,t.market
FROM sales a 
JOIN stocks t ON a.name = t.name 
WHERE to_date = '%s' AND t.status IN ("B","I", "O", "S") 
ORDER BY t.status, a.name
"""
sql = sql % yesterday
print(sql)


SELECT a.name,fm_date,to_date,fm_price,to_price,
a.qty,a.max_price,a.min_price,t.status,t.market
FROM sales a 
JOIN stocks t ON a.name = t.name 
WHERE to_date = '2023-02-06' AND t.status IN ("B","I", "O", "S") 
ORDER BY t.status, a.name



### This statement causes program to hang when there is no data

In [37]:
df = pd.read_sql(sql, conlite)
df.set_index(["name"], inplace=True)
df['fm_date'] = pd.to_datetime(df['fm_date'])
df['to_date'] = pd.to_datetime(df['to_date'])
df.eval('diff = to_price - fm_price',inplace=True)
df['percent'] = round(df['diff']/df['fm_price']*100,2)
df.style.format(format_dict)

,fm_date,to_date,fm_price,to_price,qty,max_price,min_price,status,market,diff,percent
name,,,,,,,,,,,
BANPU,2023-02-06,2023-02-06,11.50,11.30,"114,777,951",12.60,11.30,B,SET50,-0.20,-1.74%
KCE,2023-01-31,2023-02-06,54.75,56.50,"239,642,338",58.50,50.50,B,SET100,1.75,3.20%
RCL,2023-02-03,2023-02-06,30.75,31.00,"4,345,349",31.00,30.50,B,SET100,0.25,0.81%
DIF,2023-02-02,2023-02-06,13.60,13.60,"24,083,621",13.70,13.60,I,SET,0.00,0.00%
IVL,2023-02-06,2023-02-06,40.50,41.00,"16,598,179",41.25,40.25,I,SET50,0.50,1.23%
JASIF,2023-02-03,2023-02-06,8.25,8.25,"6,779,506",8.25,8.20,I,SET,0.00,0.00%
JMART,2023-02-03,2023-02-06,37.25,37.25,"7,649,901",37.75,37.00,I,SET50,0.00,0.00%
JMT,2023-02-03,2023-02-06,54.75,53.75,"18,242,178",55.25,53.75,I,SET50,-1.00,-1.83%
ORI,2023-02-06,2023-02-06,12.00,11.70,"5,653,682",12.30,11.70,I,SET100,-0.30,-2.50%


In [38]:
df.shape

(20, 11)

### IF the above count not equal number of orders, there must be something incorrect

### Create daily-sales from sales

In [39]:
df[cols].style.format(format_dict)

,fm_date,to_date,fm_price,to_price,qty,max_price,min_price,percent,status
name,,,,,,,,,
BANPU,2023-02-06,2023-02-06,11.50,11.30,"114,777,951",12.60,11.30,-1.74%,B
KCE,2023-01-31,2023-02-06,54.75,56.50,"239,642,338",58.50,50.50,3.20%,B
RCL,2023-02-03,2023-02-06,30.75,31.00,"4,345,349",31.00,30.50,0.81%,B
DIF,2023-02-02,2023-02-06,13.60,13.60,"24,083,621",13.70,13.60,0.00%,I
IVL,2023-02-06,2023-02-06,40.50,41.00,"16,598,179",41.25,40.25,1.23%,I
JASIF,2023-02-03,2023-02-06,8.25,8.25,"6,779,506",8.25,8.20,0.00%,I
JMART,2023-02-03,2023-02-06,37.25,37.25,"7,649,901",37.75,37.00,0.00%,I
JMT,2023-02-03,2023-02-06,54.75,53.75,"18,242,178",55.25,53.75,-1.83%,I
ORI,2023-02-06,2023-02-06,12.00,11.70,"5,653,682",12.30,11.70,-2.50%,I


In [40]:
file_name = "daily-sales.csv"
data_file = data_path + file_name
output_file = csv_path + file_name
box_file = box_path + file_name
one_file = one_path + file_name
osd_file = osd_path + file_name

df[cols].sort_values(['status','percent'],ascending=[True,True]).to_csv(output_file,header=None)
df[cols].sort_values(['status','percent'],ascending=[True,True]).to_csv(data_file,header=None)
df[cols].sort_values(['status','percent'],ascending=[True,True]).to_csv(box_file)
df[cols].sort_values(['status','percent'],ascending=[True,True]).to_csv(one_file)
df[cols].sort_values(['status','percent'],ascending=[True,True]).to_csv(osd_file)

In [41]:
df[cols].shape

(20, 9)

In [42]:
df.shape

(20, 11)

In [43]:
sales = df[cols]
sales

,fm_date,to_date,fm_price,to_price,qty,max_price,min_price,percent,status
name,,,,,,,,,
BANPU,2023-02-06,2023-02-06,11.50,11.30,114777951,12.60,11.30,-1.74,B
KCE,2023-01-31,2023-02-06,54.75,56.50,239642338,58.50,50.50,3.20,B
RCL,2023-02-03,2023-02-06,30.75,31.00,4345349,31.00,30.50,0.81,B
DIF,2023-02-02,2023-02-06,13.60,13.60,24083621,13.70,13.60,0.00,I
IVL,2023-02-06,2023-02-06,40.50,41.00,16598179,41.25,40.25,1.23,I
JASIF,2023-02-03,2023-02-06,8.25,8.25,6779506,8.25,8.20,0.00,I
JMART,2023-02-03,2023-02-06,37.25,37.25,7649901,37.75,37.00,0.00,I
JMT,2023-02-03,2023-02-06,54.75,53.75,18242178,55.25,53.75,-1.83,I
ORI,2023-02-06,2023-02-06,12.00,11.70,5653682,12.30,11.70,-2.50,I


In [53]:
file_name = "Price-Today.csv"
input_file = data_path + file_name
prices = pd.read_csv(input_file)
prices

,name,date,price,chg_amt,chg_pct,open,high,low,volume,daily_volume
0,ACE,2023-02-07,2.56,0.00,0.000000,2.56,2.58,2.54,23915176,61242.85760
1,ADVANC,2023-02-07,196.50,-0.50,-0.253807,197.00,198.00,196.00,3223977,633905.66650
2,AH,2023-02-07,32.75,-0.25,-0.757576,33.00,33.25,32.50,1688685,55467.10525
3,AIE,2023-02-07,3.00,0.14,4.895105,2.90,3.00,2.88,5198687,15360.20196
4,AIMIRT,2023-02-07,12.30,-0.10,-0.806452,12.40,12.40,12.30,122600,1508.11000
...,...,...,...,...,...,...,...,...,...,...
189,WHAIR,2023-02-07,8.00,0.00,0.000000,8.05,8.05,8.00,846450,6771.94520
190,WHART,2023-02-07,11.60,0.00,0.000000,11.60,11.70,11.60,484312,5621.89920
191,WHAUP,2023-02-07,4.12,0.00,0.000000,4.12,4.14,4.08,1427266,5862.84376
192,WICE,2023-02-07,11.70,-0.30,-2.500000,12.00,12.10,11.60,6565095,77325.61610


In [54]:
df_merge = pd.merge(sales,prices,on='name', how='inner')

In [55]:
df_merge

,name,fm_date,to_date,fm_price,to_price,qty,max_price,min_price,percent,status,date,price,chg_amt,chg_pct,open,high,low,volume,daily_volume
0,BANPU,2023-02-06,2023-02-06,11.50,11.30,114777951,12.60,11.30,-1.74,B,2023-02-07,11.20,-0.10,-0.884956,11.30,11.50,11.20,161516144,1.831534e+06
1,KCE,2023-01-31,2023-02-06,54.75,56.50,239642338,58.50,50.50,3.20,B,2023-02-07,57.00,0.50,0.884956,56.00,57.75,56.00,17516788,9.975574e+05
2,RCL,2023-02-03,2023-02-06,30.75,31.00,4345349,31.00,30.50,0.81,B,2023-02-07,31.00,0.00,0.000000,31.00,31.25,30.75,1864108,5.775278e+04
3,DIF,2023-02-02,2023-02-06,13.60,13.60,24083621,13.70,13.60,0.00,I,2023-02-07,13.60,0.00,0.000000,13.60,13.70,13.60,7390171,1.008499e+05
4,IVL,2023-02-06,2023-02-06,40.50,41.00,16598179,41.25,40.25,1.23,I,2023-02-07,40.50,-0.50,-1.219512,40.75,41.00,40.25,11842657,4.811257e+05
5,JASIF,2023-02-03,2023-02-06,8.25,8.25,6779506,8.25,8.20,0.00,I,2023-02-07,8.25,0.00,0.000000,8.25,8.25,8.20,3589348,2.953335e+04
6,JMART,2023-02-03,2023-02-06,37.25,37.25,7649901,37.75,37.00,0.00,I,2023-02-07,36.75,-0.50,-1.342282,37.25,37.50,36.50,8468235,3.136440e+05
7,JMT,2023-02-03,2023-02-06,54.75,53.75,18242178,55.25,53.75,-1.83,I,2023-02-07,53.00,-0.75,-1.395349,54.00,54.25,52.25,14084684,7.469663e+05
8,ORI,2023-02-06,2023-02-06,12.00,11.70,5653682,12.30,11.70,-2.50,I,2023-02-07,11.70,0.00,0.000000,11.80,11.80,11.50,8430180,9.813846e+04
9,SINGER,2023-02-03,2023-02-06,28.25,28.25,5851577,28.75,28.00,0.00,I,2023-02-07,28.00,-0.25,-0.884956,28.50,28.50,28.00,1866964,5.252393e+04


In [56]:
df_merge.columns

Index(['name', 'fm_date', 'to_date', 'fm_price', 'to_price', 'qty',
       'max_price', 'min_price', 'percent', 'status', 'date', 'price',
       'chg_amt', 'chg_pct', 'open', 'high', 'low', 'volume', 'daily_volume'],
      dtype='object')

In [57]:
colu = 'name fm_date to_date fm_price to_price qty max_price min_price percent status \
price chg_amt volume date'.split()
colu

['name',
 'fm_date',
 'to_date',
 'fm_price',
 'to_price',
 'qty',
 'max_price',
 'min_price',
 'percent',
 'status',
 'price',
 'chg_amt',
 'volume',
 'date']

In [58]:
df_merge[colu]

,name,fm_date,to_date,fm_price,to_price,qty,max_price,min_price,percent,status,price,chg_amt,volume,date
0,BANPU,2023-02-06,2023-02-06,11.50,11.30,114777951,12.60,11.30,-1.74,B,11.20,-0.10,161516144,2023-02-07
1,KCE,2023-01-31,2023-02-06,54.75,56.50,239642338,58.50,50.50,3.20,B,57.00,0.50,17516788,2023-02-07
2,RCL,2023-02-03,2023-02-06,30.75,31.00,4345349,31.00,30.50,0.81,B,31.00,0.00,1864108,2023-02-07
3,DIF,2023-02-02,2023-02-06,13.60,13.60,24083621,13.70,13.60,0.00,I,13.60,0.00,7390171,2023-02-07
4,IVL,2023-02-06,2023-02-06,40.50,41.00,16598179,41.25,40.25,1.23,I,40.50,-0.50,11842657,2023-02-07
5,JASIF,2023-02-03,2023-02-06,8.25,8.25,6779506,8.25,8.20,0.00,I,8.25,0.00,3589348,2023-02-07
6,JMART,2023-02-03,2023-02-06,37.25,37.25,7649901,37.75,37.00,0.00,I,36.75,-0.50,8468235,2023-02-07
7,JMT,2023-02-03,2023-02-06,54.75,53.75,18242178,55.25,53.75,-1.83,I,53.00,-0.75,14084684,2023-02-07
8,ORI,2023-02-06,2023-02-06,12.00,11.70,5653682,12.30,11.70,-2.50,I,11.70,0.00,8430180,2023-02-07
9,SINGER,2023-02-03,2023-02-06,28.25,28.25,5851577,28.75,28.00,0.00,I,28.00,-0.25,1866964,2023-02-07


In [59]:
df_merge[colu].shape

(20, 14)

In [60]:
df_merge.shape

(20, 19)

In [61]:
file_name = "daily-sales-prices.csv"
data_file = data_path + file_name
output_file = csv_path + file_name
box_file = box_path + file_name
one_file = one_path + file_name
osd_file = osd_path + file_name

df_merge[colu].sort_values(['name'],ascending=[True]).to_csv(output_file,header=None, index=False)
df_merge[colu].sort_values(['name'],ascending=[True]).to_csv(data_file,header=None, index=False)
df_merge[colu].sort_values(['name'],ascending=[True]).to_csv(box_file)
df_merge[colu].sort_values(['name'],ascending=[True]).to_csv(one_file)
df_merge[colu].sort_values(['name'],ascending=[True]).to_csv(osd_file)

In [ ]:
### Call ruby ruby\daily-out-new.rb